# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Model Choice

We chose a simple seq2seq model. The model consists of two recurrent neural networks, the decoder and the encoder. The encoder takes a variable lenght input and encodes it into the fixed size context space. The decoder takes the fixed size output from the encoder and predicts a variable lenght output. In our case the input of the encoder is a portuguese sentence and the output of the decoder is hopefully the according english translation.
The encoder consists of an embedding and a dropout layer and finally a gated recurrent unit(GRU) RNN. The decoder has 2 differences to the encoder. First the output needs to be converted to an actual word. Second at every step the predicted output up to this point is also fed into the GRU. During training instead of the predicted output the label is used. The implementation is largely taken from the seq2seq tutorial from pytorch. [tutorial](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html). There are placeholders in place for the attention mechanism which we dont use for our baseline model.
We chose this model because we couldnt really find non deep learning models for translation tasks with somewhat comparable results. So instead we tried to implement an easy to reason about deep learning model. 
For evaluation we use the BLEU score: [bleu](https://en.wikipedia.org/wiki/BLEU)

Note: The set up code is in the last cell at the bottom. The data preparation is mostly described in the dataset characteristics chapter. 


## Feature Selection

[Indicate which features from the dataset you will be using for the baseline model, and justify your selection.]



The dataset simply contains pairs of sentences. An english sentence and the according portuguese translation. Due to time constraints we filtered the dataset for small sentences < 8 words and only took sentences of the form "I am", "We are" and so on

## Implementation

[Implement your baseline model here.]



In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden


class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        max_len = (
            target_tensor.size(1)
            if target_tensor is not None
            else MAX_LENGTH
        )
        for i in range(max_len):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1) 
            else:
              
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

                finished |= (decoder_input.squeeze(1) == EOS_token)
                if finished.all():
                    break

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.out(output)
        return output, hidden
    

hidden_size = 128
batch_size = 32

input_lang, output_lang, train_dataloader, val_dataloader = get_dataloaders(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]



In [ ]:
from sacrebleu import corpus_bleu
encoder.eval()
decoder.eval()

predictions = []
references = []

with torch.no_grad():
    for input_tensor, target_tensor in val_dataloader:
        
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, _ = decoder(
            encoder_outputs,
            encoder_hidden
        )

        pred_ids = decoder_outputs.argmax(dim=-1)

        for pred_seq, tgt_seq in zip(pred_ids, target_tensor):

            pred_words = []
            for idx in pred_seq:
                idx = idx.item()

                if idx == EOS_token:
                    break
                pred_words.append(output_lang.index2word[idx])

            tgt_words = []
            for idx in tgt_seq:
                idx = idx.item()

                if idx == EOS_token:
                  break
                tgt_words.append(output_lang.index2word[idx])


            predictions.append(" ".join(pred_words))
            references.append(" ".join(tgt_words))
print(predictions[0:5])
print(references[0:5])
bleu = corpus_bleu(predictions, [references])
print("BLEU:", bleu.score)


sample output from the cell above:

['i m a detective', 'i m surprised to see you', 'i m a gardener', 'they re the same age', 'you re not safe']
['i m a believer', 'i m surprised to see you', 'i m a student of this university', 'they are the same age', 'you re not safe']
BLEU: 48.78984364338214

The model gets the structure of the sentence right in most cases but sometimes replaces subjects with totally different meanings. This could be improved by increasing the trainging data size which would in turn allow for more parameters in the model. Also implementing an attention mechanism could improve things.
Since there are a lot of correct translations the BLEU score is pretty solid. But this is on a dataset with very simple sentences only so we expect the score to drop on more complex datasets.

In [ ]:
with open('por.txt') as myfile:
    lines = [line.rstrip().decode("utf-8") for line in myfile]

SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")


    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')[0:2]] for l in lines]
    print(pairs[0])
    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

MAX_LENGTH = 8
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareData('eng', 'por', True)
print(random.choice(pairs))

def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)


def get_dataloaders(batch_size, val_size=0.1):
    input_lang, output_lang, pairs = prepareData('eng', 'por', True)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)

        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)

        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    # Split into train and validation
    train_input, val_input, train_target, val_target = train_test_split(
        input_ids,
        target_ids,
        test_size=val_size,
        random_state=42,
        shuffle=True
    )

    train_dataset = TensorDataset(
        torch.LongTensor(train_input).to(device),
        torch.LongTensor(train_target).to(device)
    )

    val_dataset = TensorDataset(
        torch.LongTensor(val_input).to(device),
        torch.LongTensor(val_target).to(device)
    )

    train_dataloader = DataLoader(
        train_dataset,
        sampler=RandomSampler(train_dataset),
        batch_size=batch_size
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return (
        input_lang,
        output_lang,
        train_dataloader,
        val_dataloader
    )

def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  
    plot_loss_total = 0 

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)
